# dr_evt 07: the market end to end

Notebook 06 showed the platform side: adapters that submit to a dr_evt `Simulation`
and read start and end times back. This notebook adds the client, which is the auction.
A jobs file carries one private value per platform for every job leg; a platforms file
carries node counts and public prices. Once per window the controller reads the free
nodes each platform reports, asks a mechanism which jobs go where and what they pay,
submits the winners, and checks from the timing records that every routed leg began at
the window time. The mechanism here is VCG on the exact allocation MILP.

Sections:

1. the two input files
2. one window by hand: candidates, VCG decisions, charges, submission, timings
3. the whole run through the controller and its outputs
4. the same run with one platform served over gRPC
5. the command line

In [1]:
from pathlib import Path
import os, sys, tempfile, subprocess, json
import pandas as pd

DR_EVT = Path.cwd().resolve()                        # run from learn/ or from the repository root
while not (DR_EVT / "CMakeLists.txt").exists() and DR_EVT != DR_EVT.parent:
    DR_EVT = DR_EVT.parent
assert (DR_EVT / "CMakeLists.txt").exists(), "run this notebook from learn/ inside a dr_evt checkout"
INSTALL = Path(os.environ.get("DR_EVT_INSTALL", DR_EVT / "install"))   # the cmake install prefix
OUT = Path.cwd() / "output"                                            # scratch space, gitignored
OUT.mkdir(exist_ok=True)
sys.path[:0] = [str(INSTALL / "lib" / "python"), str(DR_EVT / "python")]   # dr_evt extension, then the package

import dr_evt_market as m
from dr_evt_market.mechanisms import Vcg, build_observation, validate_decisions, submit_decisions
EXAMPLES = DR_EVT / "python" / "examples" / "market"
print("dr_evt_market", m.__version__)

dr_evt_market 0.1.0


## 1. The two input files

`platforms.csv` follows dr_evt's own `sync_systems.csv`: a system id, a node count, and
the public price per node hour. A blank `address` means the platform runs in process; an
address means a gRPC server. `jobs.csv` is a dr_evt trace with `job_id`, an optional
`leg_id`, and one `bid:<platform>` column per platform. A blank bid means the platform is
unacceptable for that leg. Rows sharing a job id are the legs of one composite job.

In [2]:
platforms_df = pd.read_csv(EXAMPLES / "market_platforms.csv")
jobs_df = pd.read_csv(EXAMPLES / "market_jobs.csv")
display(platforms_df)
jobs_df

,system_id,total_nodes,price_per_node_hour,address
0,alpha,100,1.0,NaN
1,beta,60,2.0,NaN
2,gamma,40,3.0,NaN


,job_id,job_submit_time,num_nodes,time_limit,leg_id,bid:alpha,bid:beta,bid:gamma
0,s01,0,70,120,0,100.0,NaN,NaN
1,s02,0,50,180,0,NaN,90.0,NaN
2,s03,0,40,90,0,NaN,NaN,80.0
3,s04,0,40,100,0,75.0,70.0,NaN
4,s05,30,20,80,0,35.0,30.0,25.0
5,s06,60,10,60,0,20.0,18.0,16.0
6,s07,120,60,100,0,85.0,80.0,NaN
7,s08,180,30,120,0,55.0,58.0,52.0
8,c01,240,30,100,left,65.0,60.0,55.0
9,c01,240,20,100,right,50.0,52.0,55.0


In [3]:
specs = m.read_platforms(EXAMPLES / "market_platforms.csv")
jobs, bids = m.read_jobs(EXAMPLES / "market_jobs.csv", specs)
prices = {s.system_id: s.price_per_node_hour for s in specs}
print(len(jobs), "jobs,", sum(len(j.legs) for j in jobs), "legs;", "composite:", [j.job_id for j in jobs if len(j.legs) > 1])
print("c01 bid:", {leg.leg_id: dict(leg.value_by_platform) for leg in bids["c01"].legs})

13 jobs, 14 legs; composite: ['c01']
c01 bid: {'left': {'alpha': 65.0, 'beta': 60.0, 'gamma': 55.0}, 'right': {'alpha': 50.0, 'beta': 52.0, 'gamma': 55.0}}


## 2. One window by hand

At time 0 four jobs have arrived. `build_observation` turns the queue, the bids and the
platforms' snapshots into what a mechanism sees: for every job, the placements in which
each leg fits in its platform's free nodes and its bid names that platform, with the
public resource cost of each. The private values never leave the observation's `bids`.

In [4]:
WORK = Path(tempfile.mkdtemp(dir=OUT, prefix="market07_"))
platforms = {s.system_id: m.InProcessPlatform(s.system_id, s.total_nodes, WORK / "hand" / s.system_id) for s in specs}
for p in platforms.values():
    p.advance_to(0)
queue = [j for j in jobs if j.submit_s <= 0]
obs = build_observation(0, 0, 0, queue, bids, {n: p.snapshot() for n, p in platforms.items()}, prices)
rows = []
for job in obs.jobs:
    for c in job.candidates:
        rows.append({"job": job.job_id, "placement": c.placement_id, "nodes": dict(c.demand_by_platform),
                     "value": obs.value(job.job_id, c.placement_id), "cost": round(c.resource_cost_credits, 2),
                     "net": round(obs.net_value(job.job_id, c.placement_id), 2)})
print("free nodes:", dict(obs.free_nodes))
pd.DataFrame(rows)

free nodes: {'alpha': 100, 'beta': 60, 'gamma': 40}


,job,placement,nodes,value,cost,net
0,s01,alpha,{'alpha': 70},100.0,2.33,97.67
1,s02,beta,{'beta': 50},90.0,5.00,85.00
2,s03,gamma,{'gamma': 40},80.0,3.00,77.00
3,s04,alpha,{'alpha': 40},75.0,1.11,73.89
4,s04,beta,{'beta': 40},70.0,2.22,67.78


Four jobs want 200 nodes in total and alpha, beta and gamma have 200 between them, but
`s01` and `s04` both want alpha and `s02` needs 50 of beta's 60. VCG solves the exact
welfare problem, then charges each winner the resource cost plus the welfare the others
lose because of it (the Clarke pivot). A winner that displaces nobody pays cost only.

In [5]:
decisions = Vcg().decide(obs)
accepted, rejected = validate_decisions(obs, decisions)
table = pd.DataFrame([{"job": d.job_id, "placement": d.placement_id,
                       "value": obs.value(d.job_id, d.placement_id),
                       "cost": round(obs.candidate(d.job_id, d.placement_id).resource_cost_credits, 2),
                       "charge": round(d.charge_credits, 2),
                       "pivot": round(d.charge_credits - obs.candidate(d.job_id, d.placement_id).resource_cost_credits, 2)} for d in accepted])
print("welfare of the window:", round(accepted[0].score, 2), "| rejected by validation:", [(r.decision.job_id, r.reason) for r in rejected])
table

welfare of the window: 259.67 | rejected by validation: []


,job,placement,value,cost,charge,pivot
0,s01,alpha,100.0,2.33,76.22,73.89
1,s02,beta,90.0,5.00,72.78,67.78
2,s03,gamma,80.0,3.00,3.00,0.00


In [6]:
handles = submit_decisions(accepted, obs, platforms, 0)
for p in platforms.values():
    p.advance_to(0)                                                    # evaluate the submissions at the window time
for (job_id, leg_id), handle in handles.items():
    platform = obs.candidate(job_id, next(d.placement_id for d in accepted if d.job_id == job_id)).platform_by_leg[leg_id]
    t = platforms[platform].timings([handle])[0]
    print(f"{job_id}/{leg_id} on {platform}: begin {t.begin_s:.0f} end {t.end_s:.0f} (limit {t.limit_s})")
print("waiting after submission:", {n: p.snapshot().waiting_jobs for n, p in platforms.items()})
for p in platforms.values():
    p.finish()

s01/0 on alpha: begin 0 end 120 (limit 120)
s02/0 on beta: begin 0 end 180 (limit 180)
s03/0 on gamma: begin 0 end 90 (limit 90)
waiting after submission: {'alpha': 0, 'beta': 0, 'gamma': 0}


Every winner began at the window time and no platform has a queue, which is the
invariant the controller checks after every window. The job that was not placed, `s04`,
stays in the market queue for the next window.

## 3. The whole run

`Controller` repeats that window every 60 seconds until the arrivals are exhausted and
the queue is empty, then drains every platform and joins the timing records to the
decisions. `write_outputs` produces `routed.csv`, `windows.csv`, `rejected.csv` and a
manifest with the platform statistics and the hashes of the two CSVs.

In [7]:
platforms = {s.system_id: m.InProcessPlatform(s.system_id, s.total_nodes, WORK / "run" / s.system_id) for s in specs}
report = m.Controller(platforms, prices, Vcg(), jobs, bids, window_s=60, seed=0).run()
paths = m.write_outputs(report, WORK / "run" / "out")
windows = pd.DataFrame([{"window": w.index, "time_s": w.time_s, "queued": len(w.queued), "placed": len(w.placed),
                         "welfare": round(w.welfare, 1), "revenue": round(w.revenue, 1), "free": w.free_nodes} for w in report.windows])
print("rejected at intake:", [(r.job_id, r.reason) for r in report.rejected])
windows

rejected at intake: [('s12', 'unaffordable')]


,window,time_s,queued,placed,welfare,revenue,free
0,0,0,4,3,259.7,152.0,"{'alpha': 100, 'beta': 60, 'gamma': 40}"
1,1,60,3,2,54.4,0.6,"{'alpha': 30, 'beta': 10, 'gamma': 0}"
2,2,120,2,1,83.3,75.6,"{'alpha': 80, 'beta': 10, 'gamma': 40}"
3,3,180,2,2,129.9,3.1,"{'alpha': 40, 'beta': 60, 'gamma': 40}"
4,4,240,1,1,117.5,2.5,"{'alpha': 60, 'beta': 30, 'gamma': 40}"
5,5,300,1,1,73.3,1.7,"{'alpha': 70, 'beta': 60, 'gamma': 20}"
6,6,360,1,1,48.1,1.9,"{'alpha': 60, 'beta': 60, 'gamma': 40}"
7,7,420,0,0,0.0,0.0,"{'alpha': 60, 'beta': 60, 'gamma': 15}"
8,8,480,1,1,66.8,3.2,"{'alpha': 100, 'beta': 60, 'gamma': 40}"


In [8]:
routed = pd.read_csv(paths["routed"])
routed[["job_id", "leg_id", "platform", "window_time_s", "submit_s", "begin_s", "end_s", "value_credits", "resource_cost_credits", "charge_credits"]]

,job_id,leg_id,platform,window_time_s,submit_s,begin_s,end_s,value_credits,resource_cost_credits,charge_credits
0,s01,0,alpha,0,0,0,120,100.0,2.333333,76.222222
1,s02,0,beta,0,0,0,180,90.0,5.000000,72.777778
2,s03,0,gamma,0,0,0,90,80.0,3.000000,3.000000
3,s05,0,alpha,60,30,60,140,35.0,0.444444,0.444444
4,s06,0,alpha,60,60,60,120,20.0,0.166667,0.166667
5,s07,0,alpha,120,120,120,220,85.0,1.666667,75.555556
6,s04,0,alpha,180,0,180,280,75.0,1.111111,1.111111
7,s08,0,beta,180,180,180,300,58.0,2.000000,2.000000
8,c01,left,alpha,240,240,240,340,120.0,2.500000,2.500000
9,c01,right,gamma,240,240,240,340,120.0,2.500000,2.500000


In [9]:
assert (routed.begin_s == routed.window_time_s).all(), "a routed leg did not begin at its window"
assert ((routed.charge_credits >= routed.resource_cost_credits - 1e-9) & (routed.charge_credits <= routed.value_credits + 1e-9)).all()
wait = routed.begin_s - routed.submit_s
print("legs:", len(routed), "| mean wait for a window:", round(wait.mean(), 1), "s | max:", wait.max(), "s")
print("platform statistics:")
pd.DataFrame({n: {k: round(v, 3) for k, v in r.statistics.items() if k in ("jobs_completed", "makespan", "avg_wait_time", "utilization")} for n, r in report.reports.items()}).T

legs: 13 | mean wait for a window: 16.2 s | max: 180 s
platform statistics:


,jobs_completed,utilization,avg_wait_time,makespan
alpha,7.0,0.658,0.0,450.0
beta,2.0,0.700,0.0,300.0
gamma,4.0,0.496,0.0,590.0


Waiting time here is the time from arrival to the next window boundary, since every
placed job starts at its window; the deferred ones wait one or more windows. The
platform statistics come from dr_evt itself, so utilization and makespan are the
scheduler's numbers, not the market's.

## 4. One platform over gRPC

The controller does not know which transport a platform uses. Serving alpha from a
`dr_evt_server` that the package starts gives the same `routed.csv` byte for byte.

In [10]:
with m.ServerProcess(None, WORK / "grpc" / "server") as server:
    mixed = {}
    for s in specs:
        if s.system_id == "alpha":
            mixed[s.system_id] = m.GrpcPlatform(s.system_id, s.total_nodes, server.address, WORK / "grpc" / "server", session_name="alpha")
        else:
            mixed[s.system_id] = m.InProcessPlatform(s.system_id, s.total_nodes, WORK / "grpc" / s.system_id)
    report_grpc = m.Controller(mixed, prices, Vcg(), jobs, bids, window_s=60, seed=0).run()
    paths_grpc = m.write_outputs(report_grpc, WORK / "grpc" / "out")
print("routed.csv identical across transports:", Path(paths["routed"]).read_bytes() == Path(paths_grpc["routed"]).read_bytes())
print("sha256:", paths["routed_sha256"][:16], "==", paths_grpc["routed_sha256"][:16])

routed.csv identical across transports: True
sha256: 1c1f53716b1fcf6e == 1c1f53716b1fcf6e


## 5. The command line

`python -m dr_evt_market run` does the same from a shell: it reads the two files, starts
servers for platforms with an address when asked, runs the controller and writes the
outputs. It prints the counts and the two hashes, so two runs can be compared by eye.

In [11]:
result = subprocess.run([sys.executable, "-m", "dr_evt_market", "run",
                         "--jobs", str(EXAMPLES / "market_jobs.csv"), "--platforms", str(EXAMPLES / "market_platforms.csv"),
                         "--out", str(WORK / "cli"), "--window", "60"],
                        capture_output=True, text=True, env={**os.environ, "PYTHONPATH": os.pathsep.join(sys.path[:2])})
print(result.stdout.strip())
print("exit code:", result.returncode, "| same routed hash as section 3:", paths["routed_sha256"] in result.stdout)

windows=9
routed=13
rejected=1
routed_sha256=1c1f53716b1fcf6e0cbca38f0c66884b6681f24e09ca4fec10f308c8f97ccf1d
windows_sha256=5c65dc8ba9db68206a0945851c86d689eb6df3bfe31db0bdf07067bb6bf51b47
exit code: 0 | same routed hash as section 3: True


## What comes next

Two things are missing for the experiments the market is built for. A learned
mechanism as a second `Mechanism` subclass, RegretFormer, with its regret measured by a
fixed grid search followed by gradient ascent through the deployed pipeline. And the
runtime model on streamed jobs, so that a platform can be faster than another: today
every job runs exactly its limit, on every platform.